---

# Plota Mapa de Rastreamento de Flashes do GOES-16/19 para Intervalos de Tempo Definidos

---

- `OBJETIVO`:
> Plota mapa de rastreamento de flashes. Para vários intervalos de tempo fixo contabiliza e plota as ocorrências de flashes.

- `DADOS DE ENTRADA`:
>  Tabela CSV contendo o tempo (tempo do primeiro evento do flash), latitude e longitude do flash. Exemplo de nome do arquivo: `flash_glm_goes_2020-06-30.csv`


- `DADOS DE SAÍDA`:
> 1. Figura JPG de reastreamento dos flashes. Exemplo: `GLM_04_flash_goes16_2020-06-30_restemporal_60min_dt_60min.jpg`

- `OBSERVAÇÕES`:
   > 1. Mudar os limites da imagem: lonmin, lonmax, latmin, latmax
   > 2. Muda a data: ano, mes, dia = '2020', '06', '30'

- `REALIZADO POR`:
> Enrique V. Mattos - 03/10/2025

- `ATUALIZADO POR`:
> Enrique V. Mattos - 27/04/2026
---

# **1° Passo:** Preparando ambiente

In [ ]:
# instalações
!pip install -q ultraplot cartopy salem rasterio pyproj geopandas

# monta o drive
from google.colab import drive
drive.mount('/content/drive')

# diretório raiz
dir = '/content/drive/MyDrive/PYHTON/00_GITHUB/000_CODIGOS_REFERENCIA/03_RELAMPAGOS_SATELITE_REFERENCIA'

# diretório de entrada
dir_input = f'{dir}/output/glm_diario_goes16'

# diretório de saída
dir_output = f'{dir}/output/glm_figuras_goes16'

# importa bibliotecas
import ultraplot as uplt
import cartopy.crs as ccrs
import cartopy.io.shapereader as shpreader
import pandas as pd
from datetime import timedelta, datetime
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings("ignore")

# **Plota figura**

In [ ]:
%%time
#==================================================================================================#
#                                   DEFINIÇÃO DOS LIMITES DA IMAGEM
#==================================================================================================#
# limites das latitudes e longitudes
lonmin, lonmax, latmin, latmax = -58.3, -43.5, -34.0, -21.3

# extensão da imagem [min. lon, min. lat, max. lon, max. lat]
extent = [lonmin, latmin, lonmax, latmax]

#==================================================================================================#
#                                 LEITURA DO ARQUIVO NETCDF DIÁRIO
#==================================================================================================#
# data
ano, mes, dia = '2020', '06', '30'

# leitura da planilha CSV
df_flash = pd.read_csv(f'{dir_input}/flash_glm_goes_{ano}-{mes}-{dia}.csv')

# transforma a coluna data para índice do dataframe
df_flash.set_index('time', inplace=True)

# ordena o dataframe
df_flash.sort_index(inplace=True)

# muda a coluna "time" de "2020-06-30T00:00:00.000000000" para "2020-06-30 00:00:00.000000000"
#df_flash.index = df_flash.index.astype(str).str.replace('T', ' ')

#==================================================================================================#
#                                 DEFINIÇÕES DO GRÁFICO
#==================================================================================================#
# cria moldura da figura
fig, ax = uplt.subplots(axheight=6.9, axwidth=6.8, tight=True, proj='pcarree')

# formata os eixos
ax.format(coast=False, borders=False, innerborders=False,
          labels=True, latlines=5, lonlines=5,
          latlim=(latmin, latmax), lonlim=(lonmin, lonmax),
          small='20px', large='25px',
          title=f'Rastreamento de Flashes \n',
          titleloc='l',
          titleweight='bold',
          titlecolor='bright red')

#==================================================================================================#
#                               DEFINE AS DATAS E INTERVALOS TEMPORAIS
#==================================================================================================#
# data INICIAL
anoi, mesi, diai, hori, mini = '2020', '06', '30', '12', '30'

# data FINAL
anof, mesf, diaf, horf, minf = '2020', '06', '30', '23', '30'

# frequência temporal do rastreamento [min]. Esta variável define de quanto em quanto tempo teremos os acumulados de flashes
frequencia_temporal = 60

# intervalo de acumulação dos flashes [min]
dt_acumulado = 60

# quantidade de tempos
ntimes = int((pd.to_datetime(f'{anof}{mesf}{diaf}{horf}{minf}') - pd.to_datetime(f'{anoi}{mesi}{diai}{hori}{mini}')).total_seconds() / 60 / frequencia_temporal) + 1

# extrai a data final. Exemplo: (anof, mesf, diaf, horf, minf) + 60min
data_final_str = f'{anof}-{mesf}-{diaf} {horf}:{minf}:00'
data_final = pd.to_datetime(data_final_str)
nova_data = data_final + timedelta(minutes=dt_acumulado)
ano_novo = nova_data.year
mes_novo = str(nova_data.month).zfill(2)
dia_novo = str(nova_data.day).zfill(2)
hor_novo = str(nova_data.hour).zfill(2)
min_novo = str(nova_data.minute).zfill(2)

#==================================================================================================#
#                                       DEFINE A PALETA DE CORES
#==================================================================================================#
# define a quantide de cores "cinza" e "jet"
n_cinza = int(ntimes * 0.3)  # 30% das cores em cinza
n_jet = ntimes - n_cinza     # 70% das cores em jet

# gera cores cinza (do mais claro ao mais escuro)
cores_cinza = plt.cm.Greys(np.linspace(0.3, 0.7, n_cinza))

# gera cores jet (espectro completo)
cores_jet = plt.cm.jet(np.linspace(0, 1, n_jet))

# combina as cores
cores_combinadas = np.vstack([cores_cinza, cores_jet])

# cria colormap personalizado
cmap_personalizada = LinearSegmentedColormap.from_list('cinza_jet', cores_combinadas)

# gera a paleta final de cores
cores = [cmap_personalizada(i/ntimes) for i in range(ntimes)]

#==================================================================================================#
#                                       PLOTA FIGURA
#==================================================================================================#
# loop nos intervalos de tempos
for time, data in enumerate(pd.date_range(f'{anoi}{mesi}{diai}{hori}{mini}',f'{anof}{mesf}{diaf}{horf}{minf}', freq=f'{frequencia_temporal}min')):

    # intervalo INICIAL
    intervalo_1 = str(data)

    # intervalo FINAL
    intervalo_2 = str(data + timedelta(minutes=(dt_acumulado-1), seconds=59, microseconds=999998))

    # recorta o dado para o tempo atual
    df_horario = df_flash.loc[intervalo_1:intervalo_2]

    # plota figura
    ax.scatter(df_horario['lon'].values,
               df_horario['lat'].values,
               transform = ccrs.PlateCarree(),
               marker = 'o',
               s = 6,
               cycle = cores,
               label = intervalo_1[10:16] + ' h')

# legenda
ax.legend(title='Intervalos Temporais',
          ncols=3,
          fontsize=8,
          frameon=True,
          facecolor="gray2")

# contabiliza os flashes dentro da área plotada e dentro do período total
df_temporal = df_flash.loc[f'{anoi}-{mesi}-{diai} {hori}:{mini}' : f'{ano_novo}-{mes_novo}-{dia_novo} {hor_novo}:{min_novo}']
df_filtered = df_temporal[ (df_temporal['lat'] >= latmin) & (df_temporal['lat'] <= latmax) & (df_temporal['lon'] >= lonmin) & (df_temporal['lon'] <= lonmax)]

# total de relâmpagos
ax.text(lonmin+0.3, latmin+0.3, f'Total: {df_filtered.shape[0]} relâmpagos',
        color='red', fontsize=15, weight='bold',
        bbox=dict(facecolor='yellow', edgecolor='red', boxstyle='round,pad=0.3'))

# plota contornos dos Estados
shapefile = list(shpreader.Reader('https://github.com/evmpython/Minicurso_UFMS_SEMADESC_marco_2026/raw/main/01_utils/BR_UF_2019.shp').geometries())
ax.add_geometries(shapefile, ccrs.PlateCarree(), edgecolor='grey', facecolor='none', linewidth=0.7)

# plota subtítulo
ax.text(0.000, 1.035,
        f'Satélite: GOES-16 (8km) | Sensor: GLM | Período: {anoi}-{mesi}-{diai} {hori}:{mini}h - {ano_novo}-{mes_novo}-{dia_novo} {hor_novo}:{min_novo}h | Acumulado: {dt_acumulado}min',
        transform = ax.transAxes,
        color = 'gray',
        fontsize = 9,
        verticalalignment = 'top')

# salva figura
fig.save(f'{dir_output}/GLM_04_flash_goes16_{ano}-{mes}-{dia}_restemporal_{frequencia_temporal}min_dt_{dt_acumulado}min.jpg', dpi=300)

In [ ]:
# cores cinza
cores_cinza

In [ ]:
# cores jet (arco-iris)
cores_jet

In [ ]:
# cores combinadas
cores_combinadas

In [ ]:
# imagem do cmap
cmap_personalizada

In [ ]:
# cores totais
cores

In [ ]:
# mostra os dados
df_flash

In [ ]:
# mostra os dados filtrados
df_horario